In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Custom gates
#Amitava Chakraborty - 18th Jan 2018
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

In [ ]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_backend_config import execute
from qiskit.visualization import circuit_drawer
from qiskit.quantum_info import state_fidelity
from qiskit_backend_config import BasicAer

backend = BasicAer.get_backend('unitary_simulator')

In [ ]:
from qiskit.circuit import Gate
#1. Opaque Gates
#Gate constructor need an integer number of qubits (num_qubits) and 
#an integer number of bits (num_cbits) 
#these two define the quantum and classical width of the gate
my_gate = Gate(name='opaque_gate', num_qubits=2, params=[])

#We create a register of 5 qubits
qr = QuantumRegister(4, 'q')
#Then we create blank quantum circuit
circ = QuantumCircuit(qr)
#Now we append gates using append(instruction, qargs, cargs) method of QuantumCircuit 
#It takes Instruction instance and 
#attaches it to the circuit at the specified qargs and cargs.
circ.append(my_gate, [qr[0], qr[1]])
circ.append(my_gate, [qr[2], qr[3]])
circ.append(my_gate, [qr[1], qr[2]])
circ.draw(output='mpl')

In [ ]:
#2. Composite Gates
#Composite gates and complex circuit components are constructed and managed 
#as independent QuantumCircuits 

# Build a sub-circuit
sub_q = QuantumRegister(2)
sub_circ = QuantumCircuit(sub_q, name='sub_circuit')
#Add a Hadamard gate to 0th qubit
sub_circ.h(sub_q[0])
#Add a Pauli-Z gate to 0th and 1st qubit
sub_circ.crz(1, sub_q[0], sub_q[1])
#Add a barrier
sub_circ.barrier()
#Add an identity gate
sub_circ.id(sub_q[1])
#Add a U3 gate
sub_circ.u(1, 2, -2, sub_q[0])

sub_circ.draw(output='mpl')

In [ ]:
#through the to_instruction method, 
#the individual gate outputs are converted to Instructions to be appended 
#to a target circuit at a given location.
# Convert to a gate and stick it into an arbitrary place in the bigger circuit
sub_inst = sub_circ.to_instruction()
sub_inst.label = "Sub Inst"    # Add a label to avoid the bug

#Now we construct our main circuit of 3 qubits
qr1 = QuantumRegister(3, 'q')
circ = QuantumCircuit(qr1)
#We add a Hadamard
circ.h(qr1[0])
#We add two Pauli-X gates
circ.cx(qr1[0], qr1[1])
circ.cx(qr1[1], qr1[2])
#Now we add our sub_circuit, in the form of instruction that we had created earlier
circ.append(sub_inst, [qr1[1], qr1[2]])

circ.draw('mpl')

In [ ]:
#Circuits are not immediately decomposed upon conversion to_instruction. 
#So that circuit design is possible maintaining higher levels of abstraction. 

#When desired (or before compilation)
#sub-circuits will be decomposed via the decompose method.
decomposed_circ = circ.decompose() # Doesn't modify original
decomposed_circ.draw(output='mpl')
#Decomposed quantum circuit with 7 gates.

In [ ]:
#Parameterized Gates
#Parameterization is a common feature of many quantum algorithms, 
#as well as a standard building block for constructing libraries of standard gates and 
#sub-circuits.
#Parameterized gates are used to construct a series of experiments 
#by varying the angle of a global Rz rotation over a set of entangled qubits.
from qiskit.circuit import Parameter

#The Parameter class is used to specify a place-holder for a numeric parameter
theta = Parameter('θ')
n = 5
#Create a blank QC of 5 qubits
qc = QuantumCircuit(5, 1)
#Add a Hadamard Gate at q0
qc.h(0)
#Add CX gates to the all the qubits linearly, like 0-1,1-2,2-3 etc.
for i in range(n-1):
    qc.cx(i, i+1)

#Add a barrier    
qc.barrier()
#Add a Rotation Gate using the parameter
qc.rz(theta, range(5))
#Add another barrier
qc.barrier()

#Add CX gates to the all the qubits linearly, 
#in reverse order - like 5-4,4-3 etc.
for i in reversed(range(n-1)):
    qc.cx(i, i+1)
#Add a Hadamard Gate
qc.h(0)

#Now add a measurement unit to q0
qc.measure(0, 0)

rz_circuit, rz_theta = qc, theta  # kept for the binding examples below
qc.draw(output='mpl')
#Quantum circuit with parameterized z-rotation gates.

In [ ]:
#Inspect the circuit’s parameters property.
print(qc.parameters)

In [ ]:
#Combining Parameterized Circuits
#Parameterized circuits can be composed like standard QuantumCircuits. 

#Generally, when composing two parameterized circuits, 
#the resulting circuit will be parameterized by the union of the parameters of the input circuits.

#However, parameter names must be unique within a given circuit. 

#When attempting to add a parameter whose name is already present in the target circuit:
#a) if the source and target share the same Parameter instance, 
#the parameters will be assumed to be the same and combined
#b) if the source and target have different Parameter instances, an error will be raised
phi = Parameter('phi')

sub_circ1 = QuantumCircuit(2, name='sc_1')
sub_circ1.rz(phi, 0)
sub_circ1.rx(phi, 1)

sub_circ2 = QuantumCircuit(2, name='sc_2')
sub_circ2.rx(phi, 0)
sub_circ2.rz(phi, 1)

qc = QuantumCircuit(4)
qr = qc.qregs[0]

qc.append(sub_circ1.to_instruction(), [qr[0], qr[1]])
qc.append(sub_circ2.to_instruction(), [qr[0], qr[1]])

qc.append(sub_circ2.to_instruction(), [qr[2], qr[3]])

#qc.draw(output='mpl')
qc.draw()
# The following raises an error: "QiskitError: 'Name conflict on adding parameter: phi'"
# phi2 = Parameter('phi')
# qc.u(0.1, phi2, 0.3, 0)

#Quantum circuit with 3 custom parameterized gates.

In [ ]:
#To insert a sub-circuit under a different parameterization, 
#the to_instruction method accepts an optional argument (parameter_map) which, 
#when present, will generate instructions with the source parameter replaced by a new parameter.
p = Parameter('p')
qc = QuantumCircuit(3, name='oracle')
qc.rz(p, 0)
qc.cx(0, 1)
qc.rz(p, 1)
qc.cx(1, 2)
qc.rz(p, 2)

theta = Parameter('theta')
phi = Parameter('phi')
gamma = Parameter('gamma')

qr = QuantumRegister(9)
larger_qc = QuantumCircuit(qr)
larger_qc.append(qc.to_instruction({p: theta}), qr[0:3])
larger_qc.append(qc.to_instruction({p: phi}), qr[3:6])
larger_qc.append(qc.to_instruction({p: gamma}), qr[6:9])
larger_qc.draw(output='mpl')
#Quantum circuit with 3 custom parameterized 3-qubit gates, each on their own triple of qubits.

In [ ]:
larger_qc.decompose().draw(output='mpl')
#Quantum circuit with the gates in the diagram above, decomposed.

In [ ]:
#Binding Parameters to Values
#All circuit parameters must be bound before sending the circuit to a backend. 
#This can be done in one of two ways:
#1. The bind_parameters method accepts a dictionary mapping Parameters to values, 
#and returns a new circuit with each parameter replaced by its corresponding value. 
#Partial binding is supported, in which case the returned circuit will be parameterized 
#by any Parameters which were not mapped to a value.
import numpy as np
theta_range = np.linspace(0, 2 * np.pi, 128)
print(theta_range)
# assign_parameters (formerly bind_parameters) returns a new circuit per value
bound_circuits = [rz_circuit.assign_parameters({rz_theta: theta_val}) for theta_val in theta_range]
print(bound_circuits[0].parameters)


Each bound circuit has no free parameters left (`set()`): a quantum circuit with 5 z-rotation gates.

The old `qiskit.execute` accepted a `parameter_binds` list; with Qiskit 2.x we simply run the list of bound circuits.

In the example circuit, we apply a global $R_z(	heta)$ rotation on a 5-qubit entangled state, and so expect to see oscillation in qubit-0 at $5	heta$.

In [ ]:
job = execute(bound_circuits, backend=BasicAer.get_backend('qasm_simulator'))
counts = [job.result().get_counts(i) for i in range(len(bound_circuits))]

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(theta_range,
        list(map(lambda c: c.get('0', 0), counts)),
        '.-',
        label='0')
ax.plot(theta_range,
        list(map(lambda c: c.get('1', 0), counts)),
        '.-',
        label='1')

ax.set_xticks([i * np.pi / 2 for i in range(5)])
ax.set_xticklabels(
    ['0', r'$\frac{\pi}{2}$', r'$\pi$', r'$\frac{3\pi}{2}$', r'$2\pi$'],
    fontsize=14)
ax.set_xlabel('θ')
ax.legend()
plt.show()

Two oscillating curves of the observed counts of 0 and 1 states. The two curves are off by a relative phase of pi / 5.

### Reducing Compilation Time

Compiling over a parameterized circuit prior to binding can, in some cases, significantly reduce compilation time as compared to compiling over a set of bound circuits.

(`FakeTokyo` was removed from Qiskit; `FakeAlmadenV2`, another 20-qubit fake device, is used instead.)

In [ ]:
import time
from itertools import combinations
from qiskit.compiler import transpile
from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2

fake_backend = FakeAlmadenV2()
start = time.time()
qcs = []

theta_range = np.linspace(0, 2*np.pi, 32)

for n in theta_range:
    qc = QuantumCircuit(5)

    for k in range(8):
        for i,j in combinations(range(5), 2):
            qc.cx(i,j)
        qc.rz(n, range(5))
        for i,j in combinations(range(5), 2):
            qc.cx(i,j)

    qcs.append(qc)

compiled_circuits = transpile(qcs, backend=fake_backend)

end = time.time()
print('Time compiling over set of bound circuits: ', end-start)

In [ ]:
start = time.time()
qc = QuantumCircuit(5)
theta = Parameter('theta')

for k in range(8):
    for i,j in combinations(range(5), 2):
        qc.cx(i,j)
    qc.rz(theta, range(5))
    for i,j in combinations(range(5), 2):
        qc.cx(i,j)

transpiled_qc = transpile(qc, backend=fake_backend)
bound = [transpiled_qc.assign_parameters({theta: n}) for n in theta_range]
end = time.time()
print('Time compiling over parameterized circuit, then binding: ', end-start)